## Step 1: Mount Google Drive & Check GPU

In [ ]:
from google.colab import drive
import torch

# Mount Google Drive
drive.mount('/content/drive')

# Check GPU
print("\n" + "="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✓ GPU detected: {gpu_name}")
    print(f"  Memory: {gpu_memory:.1f} GB")
else:
    print("⚠ WARNING: No GPU detected!")
    print("  Go to: Runtime → Change runtime type → GPU (T4)")
print("="*60)

## Step 2: Clone Repository

In [ ]:
# Clone the repository (dev branch with consolidation)
!git clone https://github.com/ketsiambaku/rna-motif-classification.git
%cd rna-motif-classification
!git checkout dev

print("\n✓ Repository cloned and switched to dev branch")

## Step 3: Extract Dataset from Google Drive

In [ ]:
import os
import tarfile
from pathlib import Path

# Path to your dataset in Google Drive
dataset_path = '/content/drive/MyDrive/dataset2.tar.gz'

print("="*60)
if os.path.exists(dataset_path):
    print(f"✓ Found dataset at: {dataset_path}")
    print("  Extracting... (this may take 2-3 minutes)")
    
    with tarfile.open(dataset_path, 'r:gz') as tar:
        tar.extractall('.')
    
    # Verify extraction
    if Path('dataset2').exists():
        num_files = sum(1 for _ in Path('dataset2').rglob('*.mrc'))
        print(f"\n✓ Dataset extracted successfully!")
        print(f"  Found {num_files} .mrc files in dataset2/")
    else:
        print("\n⚠ WARNING: dataset2/ folder not found after extraction")
except Exception as e:
        print(f"\n⚠ ERROR: {e}")
else:
    print(f"\n⚠ ERROR: Dataset not found at: {dataset_path}")
    print("\nPlease upload dataset2.tar.gz to your Google Drive:")
    print("  1. Go to drive.google.com")
    print("  2. Upload dataset2.tar.gz to 'My Drive'")
    print("  3. Re-run this cell")
print("="*60)

## Step 4: Install Dependencies

In [ ]:
# Install required packages
!pip install -q mrcfile biopython scikit-learn

print("✓ Dependencies installed: mrcfile, biopython, scikit-learn")

## Step 5: Train with 6-Class Consolidation (MAIN TRAINING)

This cell trains the model with:
- **6 consolidated classes** (small/large internal, bulge, hairpin)
- **Full dataset** (100% of 28,738 samples)
- **Batch size 32** (optimal for T4 GPU)
- **Early stopping** (patience=10)

Expected training time: **15-20 minutes**

In [ ]:
import subprocess
import sys

# Training command with 6-class consolidation
cmd = [
    sys.executable, 'src/train_hybrid.py',
    '--consolidate',      # Enable 6-class mode
    '--use-subset', '1.0',  # Use 100% of dataset
    '--batch-size', '32',
    '--device', 'cuda',
    '--num-workers', '2'
]

print("="*80)
print("Starting training with 6-class consolidation...")
print("Command:", ' '.join(cmd))
print("="*80 + "\n")

# Run with real-time output
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()

print("\n" + "="*80)
if process.returncode == 0:
    print("✓ Training completed successfully!")
else:
    print(f"⚠ Training exited with code: {process.returncode}")
print("="*80)

## Step 6: Download Results

In [ ]:
from google.colab import files
import shutil
import os

# Find the latest experiment folder
try:
    exp_folders = [f for f in os.listdir('experiments') if f.startswith('hybrid_phase2_1')]
    if exp_folders:
        exp_dir = max(exp_folders, key=lambda x: os.path.getmtime(os.path.join('experiments', x)))
        
        print(f"Creating zip file for: {exp_dir}")
        shutil.make_archive(f'results_{exp_dir}', 'zip', f'experiments/{exp_dir}')
        
        print(f"Downloading results...")
        files.download(f'results_{exp_dir}.zip')
        print(f"\n✓ Downloaded: results_{exp_dir}.zip")
    else:
        print("No experiment folders found. Did training complete successfully?")
except Exception as e:
    print(f"Error: {e}")

## Optional: Train 15-Class Model for Comparison

Run this cell to train the original 15-class model for comparison.

In [ ]:
import subprocess
import sys

# Training command WITHOUT consolidation (15 classes)
cmd = [
    sys.executable, 'src/train_hybrid.py',
    '--use-subset', '1.0',
    '--batch-size', '32',
    '--device', 'cuda',
    '--num-workers', '2'
]

print("="*80)
print("Starting training with 15 original classes...")
print("="*80 + "\n")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()

print("\n" + "="*80)
print(f"Training completed with exit code: {process.returncode}")
print("="*80)

## Optional: Quick Test Run (10% subset)

Use this to test the setup before running the full training.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'src/train_hybrid.py',
    '--consolidate',
    '--use-subset', '0.1',  # Only 10% of data
    '--batch-size', '32',
    '--device', 'cuda',
    '--num-workers', '2'
]

print("Running quick test with 10% of data...\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()